<div style="display: flex; align-items: center;">
  <img src="https://raw.githubusercontent.com/bartczernicki/DecisionIntelligence.GenAI.Workshop/main/Images/SemanticKernelLogo.png" width="40px" style="margin-right: 10px;">
  <span style="font-size: 1.5em; font-weight: bold;">3c - Workshop (AI Extensions) - Open Source Decision Intelligence</span>
</div>

Decision Intelligence applied in this module:  
* Listing key factors to consider when making a sound decision  
* Using OSS (open-source) reasoning models to optimize the decision approach 
* Decision Scenario: Use a decision framework (Ben Franklin's Pro & Con List) to create a decision plan  
* Improving Decision Intelligence process by explicitly providing decision frameworks and additional context 

A recommended enterprise pattern is to scale Articial Intelligence strategy with three key pillars:
* **Commercial AI** (OpenAI and other proprietary Generative AI providers)
* **Open Source (OpenWeight) AI** (open-source AI providers)
* **Vendor and Partner AI** (i.e. company HR Software, contract software) 

> 📝 Note: There is a technical difference between Open Source and Open Weight models. Open Weight models generally only provide you the weights & biases file(s) with a friendly open source license. Open Source models include the pre/post training data, training scripts and sometimes the training checkpoints. Those three additions allow the models to be fully retrained by anyone. More info can be found here: https://opensource.org/ai/open-weights. For this workshop, we will use the term open source more loosely. A great majority of the cases most organizations just want two things: the weights & biases file(s) and a friendly licnese. Certain organizations certainly require full model assets for risk management. While the full open source model assets are nice to have, they offer minimal value in most cases. 

These three pillars (listed above) strategically form AI capability and capacity in what I like to refer to as the **"Generative AI Brain"**. This is illustrated below with sample providers. Most organizations scale their AI investments with Commercial AI providers, such as: OpenAI, Anthropic or Google. However, some organizations prefer to have more governance controls over their AI and prefer open-source alternatives. For hobby consumers, open source translates to no credit card requirements for APIs and very high privacy. 
Note that there are some commercial providers like Meta or Cohere that provide open weight models. Finally in the third tier, an organization may have an existing relationship with a vendor/partner that has their own AI integrations. For example, Adobe offers enterprise graphic design/contract software and leveraging their built-in Generative AI capabilities rather building their own can make a lot of sense. 

<img style="display: block; margin: auto;" width ="600px" src="https://raw.githubusercontent.com/bartczernicki/DecisionIntelligence.GenAI.Workshop/main/Images/AIBrainPillars.png">

Most developer AI frameworks work across all the pillars mentioned above. It allows all types of models (commercial or proprietary) and almost any APIs to be orchestrated to faciliate enterprise Decision Intelligence. This is a much deeper subject and for simplicity this module highlights how to use local GenAI models (i.e. LLMs) with Microsoft AI for decision-making. Models from the OpenAI OSS family will be selected to run locally using LMStudio as a local REST endpoint that will interface with Microsoft AI orchestration.  

Open Source (OSS) models vary dramatically in size. They can have a small number of parameters/activated layers and perform great locally on your mobile device! They can also have a huge number parameters that rivals commercial AI LLMs. 

In the cases where OSS models have a small number of parameters, they are are considered SLMs (Small Language Models) with parameters generally below the 30 Billion parameter threshhold. This allows most of these models to run comfortably on commodity hardware available to personal users. This doesn't mean these models are not capable of performing well. While these models certainly may lack the general knowledge breadth of frontier AI Large Language Models, SLMs make up for it by providing very specialized logic, math and reasoning capabilities. For example, an SLM only trained for a specific language (English) and a certain domain (legal industry) can include only the relevant training information. Therefore, it can be offered as a much more compact model with many less parameters. 

In the cases where OSS models have a large number of parameters, they can perform just as well as commercial models! OSS models with large parameters require enterprise commmerical hardware to execute at scale. Below is an image from an independent AI benchmarking site ArtificialAnalysis.ai across proprietary and OSS models. Notice that almost half of the Top 30 performing models are open weight models (blue): 

<img style="display: block; margin: auto;" width ="600px" src="https://raw.githubusercontent.com/bartczernicki/DecisionIntelligence.GenAI.Workshop/refs/heads/main/Images/ArtificialAnalysis-ModelBenchmark-202604.png">  

For this workshop, we will be working primarily with SLM family of OSS models, because not everyone has H100 or B200 Nvidia GPUs to deploy huge parameter models. However, as you will see these smaller models are quite good. As of April 2026, OpenAI's gpt-oss-20 or Gemma 4 open-source reasoning models performs about 2 generations back of frontier LLM performance. Keep in mind these are general 20B & 26B parameter OSS models. This model can be fine-tuned or mid-trained and can perform much better. 

<img style="display: block; margin: auto;" width ="600px" src="https://raw.githubusercontent.com/bartczernicki/DecisionIntelligence.GenAI.Workshop/refs/heads/main/Images/LMStats-Benchmark-202604.png">  

----
### Step 1 - Get Started with LMStudio and Local Open Source AI Models 

Steps to get started:
* Download & install the latest LMStudio version: https://lmstudio.ai/ (Windows, Mac or Linux) 
* Run the LMStudio studio application.
* In the LMStudio application, search for **Gemma 4 26B A4B** GGUF (MLX is optimized on macOS) in the "Discover" section of LMStudio. A variety of Gemma options that are official and unofficial from hobbyists will appear. Typically, selecting the official model with the most downloads will provide the best results. In this case, you can be safe by selecting the official **Google** provider. You can select different quantizations of the model, to optimize the performance. 
* In the experiment below, the **26B parameter model is being used with 4bit quantization** is selected. LMStudio will inspect your hardware and let you know which quantized version of the model(s) is optimal for your hardware. Note: Even computers with small graphics cards can run these models well locally. Furthermore, laptops such as the Macbook Pro with Neural Engine can run LMStudio local models as well. You just need memory. 
* Start the LMStudio Server with the **Gemma 4 26B A4B** model loaded. This will start a local REST endpoint with a URI similar to http://10.0.0.18:1234/v1 
* The LMStudio local server does not have default security, you can simply check by navigating to this link in any browser to check if a model is loaded: **http://10.0.0.18:1234/v1/models** in the web browser. 

<img style="display: block; margin: auto;" width ="600px" src="https://raw.githubusercontent.com/bartczernicki/DecisionIntelligence.GenAI.Workshop/main/Images/LMStudioServer.png"> 

----
### Step 2 - Initialize ChatClient using OpenAI libraries

Execute the next cell to:
* Use the Configuration Builder to use the local LMStudio Server  
* Use the local API configuration to build an API compatible OpenAIClient
* The API Compatible OpenAIClient can be converted to a Microsoft.Extensions.AI ChatClient abtraction
* Note: Notice there is no security being passed in and it is simply a URL

In [1]:
// Install the required AI packages
#r "nuget: Microsoft.Extensions.DependencyInjection, 10.0.6"
#r "nuget: Microsoft.Extensions.AI, 10.5.0"
#r "nuget: Microsoft.Extensions.AI.Abstractions, 10.5.0"
#r "nuget: Microsoft.Extensions.AI.OpenAI, 10.5.0"
#r "nuget: OpenAI, 2.10.0"

using Microsoft.Extensions.AI;
using OpenAI;
using System.ClientModel; // used by APiKeyCredential class


var localApiKey = "not_needed_for_lmstudio_ollama_etc"; // local LLMs do not require an API key
var localUrl = "http://10.0.0.61:1234/v1/"; // this can be localhost or an IP address
var localModelName = "google/gemma-4-26b-a4b"; // Another Option: "openai/gpt-oss-20b";

var apiCredentials = new ApiKeyCredential(localApiKey);
var openAIClientOptions = new OpenAIClientOptions
{
    Endpoint = new Uri(localUrl)
};

// Create a local AI client 
var localAIClient = new OpenAIClient(apiCredentials, openAIClientOptions);

// Wrap the OpenAI-compatible chat client with the Microsoft.Extensions.AI abstraction.
IChatClient localAIChatClient = localAIClient
    .GetChatClient(localModelName)
    .AsIChatClient();

The below script needs to be able to find the current output cell; this is an easy method to get it.

Installed Packages Microsoft.Extensions.AI, 10.5.0 Microsoft.Extensions.AI.Abstractions, 10.5.0 Microsoft.Extensions.AI.OpenAI, 10.5.0 Microsoft.Extensions.DependencyInjection, 10.0.6 OpenAI, 2.10.0

---
### Step 3 - Open Source AI with Decision Intelligence 

The OpenAI .NET library allows one to interact with any API service that adheres to the OpenAI specification. Notice the method to add LMStudio capability was simply enabled via the **AddOpenAIChatCompletion** method above. 

Execute the cell below about decision factors for a investment property. Note:
* OpenAI Prompt Execution Settings are the same in LMStudio as they are for OpenAI and Azure OpenAI
* OSS models have specific model cards identifying best practices 
* OSS reasoning models will output their reasoning tokens inside the <think> XML tags
* Passing in arguents works the same way in OpenAI as other models (i.e. Temperature, Reasoning Effort etc.)

> 📝 **Note:** It is important to note that different model providers (commercial and open-source) have different recommended general settings that need to be tested. While different types of AI systems will require optimizing these settings, recommended settings are a good starting point.

In [3]:
// Define the system prompt for the Decision Intelligence assistant
var systemDecisionPrompt = """
You are a Decision Intelligence assistant. 
Assist the user in exploring options, reasoning through decisions, problem-solving, and applying systems thinking to various scenarios. 
Provide structured, logical, and comprehensive advice.

Output Formatting Instructions:
When generating Markdown, do not use any headings higher than ###. 
Avoid # and ## headers. Use only ###, ####, or lower-level headings if necessary. 
All top-level section headers should start at ### or lower. 
Never use ---, ***, or ___ for horizontal lines. There should be no horizontal lines in the output.
For separation, use extra extra spacing. Do not any render horizontal lines.

Format the response using only a Markdown table. Only return a Markdown table. 
Do not enclose the table in triple backticks.
""";

// Create a Decision Intelligence prompt on the topic of purchasing a secondary home as an investment property
// Provide detailed decision-making criteria for evaluating the investment decision
var simpleDecisionPrompt = """
You are considering purchasing a secondary home as an investment property. 

What key factors should you evaluate to ensure a sound investment decision, including financial, 
market, and property-specific considerations? 
Outline the critical steps and criteria for assessing location, potential rental income, 
financing options, long-term property value, and associated risks. 
""";

List<ChatMessage> chatMessages =
[
    new(ChatRole.System, systemDecisionPrompt),
    new(ChatRole.User, simpleDecisionPrompt),
];

// Define reasoning options for the chat completion.
// Note: For speed optimization, reasoning effort is set to None.
var reasoningOptions = new ReasoningOptions
{
    Effort = ReasoningEffort.None
};

// Note: Different OSS models have different capabilities and settings
// Gemma-4-26B Model Card Recommendations: https://huggingface.co/google/gemma-4-26B-A4B#1-sampling-parameters 
ChatOptions chatOptions = new()
{
    Reasoning = reasoningOptions
};

// Execute the chat messages through the Microsoft.Extensions.AI IChatClient API.
ChatResponse chatHistoryResponse = await localAIChatClient.GetResponseAsync(chatMessages, chatOptions);
var chatHistoryResponseText = chatHistoryResponse.Text;

// Display the response string as Markdown
chatHistoryResponseText.DisplayAs("text/markdown");

| Category | Key Evaluation Factor | Critical Steps & Assessment Criteria |
| :--- | :--- | :--- |
| Financial Considerations | Cash Flow Analysis | Calculate Net Operating Income (NOI) by subtracting all operating expenses (taxes, insurance, maintenance, property management) from gross rental income. Ensure positive monthly cash flow after debt service. |
| | Return on Investment (ROI) | Calculate both Cash-on-Cash Return (annual pre-tax cash flow divided by total cash invested) and Cap Rate (NOI divided by purchase price) to compare against other asset classes. |
| | Financing & Leverage | Evaluate mortgage terms for secondary residences (often higher interest rates/down payments). Factor in debt-to-income ratios and ensure sufficient liquidity for emergency reserves. |
| Market Considerations | Location Micro-Analysis | Assess proximity to transit, employment hubs, schools, and amenities. Use "walkability scores" and crime statistics to determine demand stability. |
| | Market Trends & Absorption | Analyze historical price appreciation in the specific zip code and current vacancy rates. Determine if the market is growing, stagnant, or declining. |
| | Demand Drivers | Identify local economic drivers (e.g., a new hospital, university, or corporate headquarters) that guarantee a steady influx of potential tenants. |
| Property-Specific Factors | Physical Condition & CapEx | Conduct thorough inspections to identify immediate repair needs and long-term Capital Expenditures (e.g., roof, HVAC, foundation). Budget for these in your initial financial model. |
| | Rental Yield Potential | Compare the property's size and features against local "comps" (comparable rentals) to ensure the asking rent is realistic for the area. |
| | Scalability & Versatility | Evaluate if the property can be easily reconfigured (e.g., adding a bedroom) or if it has multi-unit potential to increase income later. |
| Long-Term Strategy | Appreciation Potential | Research zoning laws and planned local developments that could increase land value over a 5-10 year horizon. |
| | Exit Strategy | Determine the ease of resale. Is the property a niche luxury item (harder to liquidate) or a standard residential type (easier to liquidate)? |
| Risk Management | Vacancy Risk | Stress-test your financials by simulating 10% to 20% vacancy periods. Ensure you can cover mortgage payments during months without rental income. |
| | Regulatory & Tax Risk | Investigate local short-term rental (STR) laws, landlord-tenant regulations, and property tax reassessment policies. |
| | Interest Rate Risk | If using variable-rate financing, model how a rate hike would impact your ability to maintain positive cash flow. |

---
### Step 4 - Open Source AI with Decision Intelligence (Advanced)

<img style="display: block; margin: auto;" width ="600px" src="https://raw.githubusercontent.com/bartczernicki/DecisionIntelligence.GenAI.Workshop/refs/heads/main/Images/DecisionIntelligence-Quote-WillRodgers-RealEstate.png"> 

Advanced Prompt Engineering techniques can be applied to OSS (open-source) models as well. In the example below a more advanced reasoning decision prompt will be used to provide additional instructions to the GenAI model. Reasoning models do a nice job in approaching the problem with an inner monologue, however you can provide additional instructions for the model to consider as they are thinking about an approach.  

In [4]:
// Define the system prompt for the Decision Intelligence assistant
var systemDecisionPrompt = """
You are a Decision Intelligence assistant. 
Assist the user in exploring options, reasoning through decisions, problem-solving, and applying systems thinking to various scenarios. 
Provide structured, logical, and comprehensive advice.

Output Formatting Instructions:
When generating Markdown, do not use any headings higher than ###. 
Avoid # and ## headers. Use only ###, ####, or lower-level headings if necessary. 
All top-level section headers should start at ### or lower. 
Never use ---, ***, or ___ for horizontal lines. There should be no horizontal lines in the output.
For separation, use extra extra spacing. Do not any render horizontal lines.

Format the response using only a Markdown table. Only return a Markdown table. 
Do not enclose the table in triple backticks.
""";

// Create a Decision Intelligence prompt on the topic of purchasing a secondary home as an investment property
// Use Chain of Thought to prompt the OSS model
// Use the Minto Pyramid to communicate the decision 
var advancedDecisionPrompt = """
You are considering purchasing a secondary home as an investment property. 

Before providing any answer, in your reasoning process consider the following:
Understand the Problem: Carefully read and understand the user's question or request. 
Break Down the Reasoning Process: Outline the steps required to solve the problem or respond to the request logically and sequentially. Think aloud and describe each step in detail. 
Always aim to make your thought process transparent and logical. 
Explain Each Step: Provide reasoning or calculations for each step, explaining how you arrive at each part of your answer. 
Provide structured, logical, and comprehensive advice. 
Arrive at the Final Answer: Only after completing all steps, provide the final answer or solution. 
Review the Thought Process: Double-check the reasoning for errors or gaps before finalizing your response. 
Communicate the final decision using the Minto Pyramid Principle.
""";

List<ChatMessage> chatMessagesDecision =
[
    new(ChatRole.System, systemDecisionPrompt),
    new(ChatRole.User, advancedDecisionPrompt),
];

// Define reasoning options for the chat completion.
// Note: For speed optimization, reasoning effort is set to None.
var reasoningOptions = new ReasoningOptions
{
    Effort = ReasoningEffort.None
};

// Note: Different OSS models have different capabilities and settings
// Gemma-4-26B Model Card Recommendations: https://huggingface.co/google/gemma-4-26B-A4B#1-sampling-parameters 
ChatOptions chatOptions = new()
{
    Temperature = 1.0f,
    TopP = 0.95f,
    TopK = 64,
    Reasoning = reasoningOptions
};

// Execute the chat messages through the Microsoft.Extensions.AI IChatClient API.
ChatResponse chatHistoryAdvancedDecisionPromptResponse = await localAIChatClient.GetResponseAsync(chatMessagesDecision, chatOptions);
var chatHistoryAdvancedDecisionPromptText = chatHistoryAdvancedDecisionPromptResponse.Text;

// Render the response string as Markdown
chatHistoryAdvancedDecisionPromptText.DisplayAs("text/markdown");

// Add the assistant's response to the chat history
// This allows you to maintain the context of the conversation for future messages
chatMessages.Add(new ChatMessage(ChatRole.Assistant,chatHistoryAdvancedDecisionPromptText));

| Dimension | Key Considerations & Decision Criteria | Strategic Questions to Ask |
| :--- | :--- | :--- |
| **Financial Viability** | Cash flow analysis (rental income vs. mortgage/taxes/maintenance), Return on Investment (ROI), Capital Appreciation potential, and Opportunity Cost of capital. | Will the monthly net cash flow be positive? Does the projected appreciation outperform a passive index fund? |
| **Market Analysis** | Local demand for rentals, seasonal occupancy trends, employment growth in the area, and school district quality. | Is this a high-growth area or a saturated market? Is the property suitable for long-term vs. short-term rental models? |
| **Risk Management** | Interest rate fluctuations, vacancy rates, property damage/maintenance, regulatory changes (e.g., STR bans), and liquidity risks. | How much of a "safety buffer" exists if the property remains vacant for three months? Are local laws favorable to investors? |
| **Operational Complexity** | Property management requirements, tax implications (depreciation, capital gains), and physical upkeep/maintenance. | Do I want to be a landlord or hire a manager? How does this impact my tax liability and current debt-to-income ratio? |
| **Strategic Fit** | Alignment with long-term wealth goals, lifestyle implications (if using as a vacation home), and diversification of existing portfolio. | Does this asset class complement my current holdings? Am I over-leveraged in real estate? |

---
### Step 5 - Open Source AI with The Ben Franklin Decision Framework 

> 📜 **_"By failing to prepare, you are preparing to fail."_**
>
> -- <cite>Ben Franklin (Founding Father of the United States, inventor, godfather of Decision Science)</cite> 

<img style="display: block; margin: auto;" width ="700px" src="https://raw.githubusercontent.com/bartczernicki/Articles/main/20230326-Make-Great-Decisions-Using-Ben-Franklins-Pros-And-Cons-Method/Image-BenFranklinDecisionMakingMethod.png">

#### Tom Brady's use of a Decision Framework

<img style="display: block; margin: auto;" width ="600px" src="https://raw.githubusercontent.com/bartczernicki/DecisionIntelligence.GenAI.Workshop/refs/heads/main/Images/DecisionIntelligence-Quote-TomBrady-ProsAndConsList.png">

Tom Brady's decision to join the Tampa Bay Buccaneers in 2020 marked a significant in his legendary NFL career. After 20 seasons and six Super Bowl championships with the New England Patriots, Brady became a free agent and chose to sign with the Bucs. How did he arrive at this decision? On the Fox broadcast on 09.29.2024, while covering the Buccaneers vs Philadelphia Eagles game, Tom Brady described how he arrived at this decision.

In the screenshot below, Tom Brady is holding up some small paper cards he is showing the audience of the broadcast. Brady mentioned he wrote down the personal decision criteria that was important and how each team compared in that criteria (salary, weather etc). He used this to select the Tampa Bay Buccaneers as his team, where he went on to win a Super Bowl in his first year there! **After 250 years since it's inception, Tom Brady used the "Ben Franklin Decision Framework" to decide where to play NFL quaterback!!**  

<img style="display: block; margin: auto;" width ="700px" src="https://raw.githubusercontent.com/bartczernicki/DecisionIntelligence.GenAI.Workshop/main/Images/Scenarios/OpenSourceDecisionIntelligence-BenFranklinDecisionFramework-TomBrady.png">

#### Steps for Ben Franklin's Decision Framework

Below are the steps Ben Franklin recommends when making a decision, which he called his "Decision Making Method of Moral Algebra":  
- Frame a decision that has two options (Yes or a No)
- Divide an area into two competing halves: a "Pro" side and "Con" side
- Label the top of one side "Pro" (for) and the other "Con" (against)
- Under each respective side, over a period of time (Ben Franklin recommended days, this could be minutes) write down various reasons/arguments that support (Pro) or are against (Con) the decision
- After spending some time thinking exhaustively and writing down the reasons, weight the different Pro and Con reasons/arguments
- Determine the relative importance of each reason or argument. This is done by taking reasons/arguments that are of similar value (weight) and crossing them off of the other competing half. Multiple reasons can be combined from one side to form a "subjective" value (weight) to balance out the other half. (For example, two medium "Pro" reasons might add up to an equal value of a single important "Con" reason)
- The side with the most remaining reasons is the option one should select for the decision in question

Learn more about Ben Franklin's Decision Framework: https://medium.com/@bartczernicki/make-great-decisions-using-ben-franklins-decision-making-method-c7fb8b17905c  

#### Decision Scenario - Should a Family Decide to Take a Luxury Vacation?

Should a family take a luxury family vacation this year? Just like Brady mapped out whether joining the Bucs would satisfy his key personal and professional goals, you can list the factors that matter most for your family—budget, timing, destination climate, activities for the kids—and lay them out on your own “decision cards.” Weigh each component carefully, just as Brady weighed his NFL future. Because if it worked to land Brady in Tampa (where he won yet another Super Bowl), imagine what it can do for a family’s dream getaway.

<img style="display: block; margin: auto;" width ="700px" src="https://raw.githubusercontent.com/bartczernicki/DecisionIntelligence.GenAI.Workshop/main/Images/Scenarios/Scenario-Vacation.png">

In [5]:
// Define the system prompt for the Decision Intelligence assistant
var systemDecisionPrompt = """
You are a decision intelligence assistant. 
Your role is to guide users in exploring options, analyzing decisions, solving complex problems, and applying systems thinking to diverse scenarios. 
Provide structured, logical, and comprehensive advice, ensuring clarity, depth, and actionable insights to support informed decision-making. 

Output Formatting Instructions:
When generating Markdown, do not use any headings higher than ###. 
Avoid # and ## headers. Use only ###, ####, or lower-level headings if necessary. 
All top-level section headers should start at ### or lower.  

Format the response using only a Markdown table. Only return a Markdown table. 
Do not enclose the table in triple backticks.
""";
var benFranklinLuxuryVacationDecisionPrompt = """
Apply the Ben Franklin Decision-Making Framework (Pro and Con list) to evaluate whether or not to take a luxury family vacation. 
List at most 5 pros and at most 5 cons to help the user make an informed decision.
""";

List<ChatMessage> chatMessagesLuxuryVacationDecision =
[
    new(ChatRole.System, systemDecisionPrompt),
    new(ChatRole.User, benFranklinLuxuryVacationDecisionPrompt),
];

// Define reasoning options for the chat completion.
// Note: For speed optimization, reasoning effort is set to None.
var reasoningOptions = new ReasoningOptions
{
    Effort = ReasoningEffort.None
};

// Note: Different OSS models have different capabilities and settings
// Gemma-4-26B Model Card Recommendations: https://huggingface.co/google/gemma-4-26B-A4B#1-sampling-parameters 
ChatOptions chatOptions = new()
{
    Temperature = 1.0f,
    TopP = 0.95f,
    TopK = 64, 
    Reasoning = reasoningOptions
};

// Execute the chat messages through the Microsoft.Extensions.AI IChatClient API.
ChatResponse chatHistoryAdvancedDecisionPromptResponse = await localAIChatClient.GetResponseAsync(chatMessagesLuxuryVacationDecision, chatOptions);
var chatHistoryAdvancedDecisionPromptText = chatHistoryAdvancedDecisionPromptResponse.Text;

// Render the response string as Markdown
chatHistoryAdvancedDecisionPromptText.DisplayAs("text/markdown");

// Add the assistant's response to the chat history
// This allows you to maintain the context of the conversation for future messages
chatMessages.Add(new ChatMessage(ChatRole.Assistant,chatHistoryAdvancedDecisionPromptText));

| Category | Description | Impact Level | Key Consideration |
| :--- | :--- | :--- | :--- |
| **Pro** | Strengthened Family Bonds | High | Shared unique experiences can create lasting emotional connections and memories. |
| **Pro** | Mental Decompression | Medium | A total break from daily stressors allows for mental resets and reduced burnout. |
| **Pro** | Cultural Enrichment | Medium | Exposure to new environments, cuisines, and traditions provides educational value. |
| **Pro** | Reward/Milestone Celebration | Medium | Serves as a meaningful way to celebrate significant life achievements or milestones. |
| **Pro** | Life Perspective Shift | Low | Breaking routine helps gain clarity on life priorities and personal values. |
| **Con** | Significant Financial Strain | High | The high cost could impact long-term savings, investments, or emergency funds. |
| **Con** | Post-Vacation Stress | Medium | Returning to reality (work/school) can cause a "re-entry" slump or logistical chaos. |
| **Con** | Opportunity Cost | High | Funds used here are unavailable for other significant life goals or investments. |
| **Con** | Travel Fatigue/Conflict | Medium | High-intensity travel can lead to physical exhaustion or interpersonal friction. |
| **Con** | Distraction from Goals | Low | Temporary removal from daily routines might delay progress on ongoing projects. |

#### Improving the Ben Franklin's Decision Framework with SLMs

For those familiar with the Ben Franklin decision framework, the output from the AI model above may not be exactly what most would anticipate. The Ben Franklin framework could be partially understood by the AI process nor fully applied. Open-Source GenAI models that have a small amount of parameters (< ~27 billion parameters) may not have all the inherent Decision Intelligence knowledge "trained" into the model. The exception being domain-specific models that are specifically trained on data sets for that domain. These domain-specific models can fill their "limited knowledge" with information that is pertinent to the tasks, while maintaining a small amount of parameters. Therefore, you could train small AI models that specialize in Decision Intelligence. 

One simple way to improve the outcome is to provide the explicit steps of the "Ben Franklin Decision Framework" into the prompt context. This basically provides the instructions of the decision framework directly to the model; regardless if the GenAI model was trained with decision framework data. By doing this extra explicit step, there is no ambiguity for the AI model how to approach the decision process. 

In the example below, the prompt context is provided with the Ben Franklin Decision Framework steps. Contrast this with the example above, where the decision recommendation is not clear. 

In [6]:
// Define the system prompt for the Decision Intelligence assistant
var systemDecisionPrompt = """
You are a decision intelligence assistant. 
Your role is to guide users in exploring options, analyzing decisions, solving complex problems, and applying systems thinking to diverse scenarios. 
Provide structured, logical, and comprehensive advice, ensuring clarity, depth, and actionable insights to support informed decision-making.

Output Formatting Instructions:
When generating Markdown, do not use any headings higher than ###. 
Avoid # and ## headers. Use only ###, ####, or lower-level headings if necessary. 
All top-level section headers should start at ### or lower.  
Never use ---, ***, or ___ for horizontal lines. There should be no horizontal lines in the output.
For separation, use extra extra spacing. Do not any render horizontal lines.

Format the response using only a Markdown table. Only return a Markdown table. 
Do not enclose the table in triple backticks.
""";

var explicitBenFranklinDecisionPrompt = """
Apply the following steps IN ORDER of the Ben Franklin Decision Framework to the Question below:
1) Frame a decision that has two options (Yes or a No)
2) Divide an area into two competing halves: a "Pro" side and "Con" side
3) Label the top of one side "Pro" (for) and the other "Con" (against)
4) Under each respective side, list a maximum of 5 reasons or arguments for each option. If there less than 5 reasons, only list the actual number of reasons there are for each side. Do not list filler reasons to reach 5 if there are not actually 5 reasons for that side.
5) Consider the weight of each reason or argument. This is done by taking reasons/arguments that are of similar value (weight) and crossing them off of the other competing half. Multiple reasons can be combined from one side to form a "subjective" value (weight) to balance out the other half. (For example, two medium "Pro" reasons might add up to an equal value of a single important "Con" reason)
6) Determine the relative importance of each reason or argument. This is done by taking reasons/arguments that are of similar value (weight) and crossing them off of the other competing half. Multiple reasons can be combined from one side to form a "subjective" value (weight) to balance out the other half. (For example, two medium "Pro" reasons might add up to an equal value of a single important "Con" reason)
7) The side with the most remaining reasons is the option one should select for the decision in question

IMPORTANT: ALWAYS recommend a decision based on the side with the most remaining reasons, even if the reasons are of lesser value than the other side!

Question: Should I take a luxury family vacation?
""";

List<ChatMessage> chatMessagesLuxuryVacationDecisionImproved =
[
    new(ChatRole.System, systemDecisionPrompt),
    new(ChatRole.User, explicitBenFranklinDecisionPrompt),
];

// Define reasoning options for the chat completion.
// Note: For speed optimization, reasoning effort is set to None.
var reasoningOptions = new ReasoningOptions
{
    Effort = ReasoningEffort.None
};

// Note: Different OSS models have different capabilities and settings
// Gemma-4-26B Model Card Recommendations: https://huggingface.co/google/gemma-4-26B-A4B#1-sampling-parameters 
ChatOptions chatOptions = new()
{
    Temperature = 1.0f,
    TopP = 0.95f,
    TopK = 64, 
    Reasoning = reasoningOptions
};

// Execute the chat messages through the Microsoft.Extensions.AI IChatClient API
ChatResponse chatLuxuryVacationDecisionImprovedResponse = await localAIChatClient.GetResponseAsync(chatMessagesLuxuryVacationDecisionImproved, chatOptions);
var chatLuxuryVacationDecisionImprovedResponseText = chatLuxuryVacationDecisionImprovedResponse.Text;

// Render the response string as Markdown
chatLuxuryVacationDecisionImprovedResponseText.DisplayAs("text/markdown");

| Step | Description/Content |
| :--- | :--- |
| 1. Decision Frame | Should I take a luxury family vacation? (Yes or No) |
| 2-4. Argument Listing | **Pro**<br>1. High-quality family bonding time.<br>2. Stress reduction and mental rejuvenation.<br>3. Creating lifelong memories for children.<br>4. Reward for recent hard work/achievements.<br><br>**Con**<br>1. Significant financial cost and impact on savings.<br>2. Logistical stress of planning/traveling with family. |
| 5-6. Weighting & Relative Importance | **Analysis Process:**<br>To balance the sides, we group the Pro arguments: The combination of bonding, rejuvenation, memories, and reward (4 medium reasons) is weighted against the significant financial cost (1 heavy Con reason). The logistical stress (1 medium Con reason) is weighed against the remaining Pro value.<br><br>**Resulting Tally:**<br>After cross-referencing the weight of the financial impact against the combined emotional and experiential benefits, 3 Pro reasons remain effective, while only 1 Con reason remains significant. |
| 7. Final Recommendation | **Decision: Yes**<br>The Pro side contains the most remaining reasons after the weighting process. |

The GenAI model may or may not recommend a luxury vacation depending on the executed run. It's decision response is highly generic and isn't grounded on personal information that can influence the decision recommendation. This can be dramatically improved further! Imagine if the GenAI model had access to: your finances, current stress level, the last time you took a vacation, any upcoming major purchases, family dynamic?!  

In the optimized decision example below, additional context is provided with that information. Notice how it changes the the Pro and Con list.  

Just like Tom Brady, the AI could craft a Pro and Con list specific and personalized to your scenario!

In [7]:
// Define the system prompt for the Decision Intelligence assistant
var systemDecisionPrompt = """
You are a decision intelligence assistant. 
Your role is to guide users in exploring options, analyzing decisions, solving complex problems, and applying systems thinking to diverse scenarios. 
Provide structured, logical, and comprehensive advice, ensuring clarity, depth, and actionable insights to support informed decision-making.

Output Formatting Instructions:
When generating Markdown, do not use any headings higher than ###. 
Avoid # and ## headers. Use only ###, ####, or lower-level headings if necessary. 
All top-level section headers should start at ### or lower.  
Never use ---, ***, or ___ for horizontal lines. There should be no horizontal lines in the output.
For separation, use extra extra spacing. Do not any render horizontal lines.

Format the response using only a Markdown table. Only return a Markdown table. 
Do not enclose the table in triple backticks.
""";
// Try changing the background information to see how it affects the decision-making process
var backroundInformation = """
BACKGROUND INFORMATION OF FAMILY: 
You are considering purchasing a secondary home as an investment property. 
You have been stressed out at work. 
You have been working long hours and have not taken a vacation in over a year. 
You have received a recent promotion (pay raise with a large bonus coming in a few months).
Your car is finishing its lease and will need to be replaced soon. 
""";
// Try to adjust the specificity of the decision-making criteria to see how it affects the decision-making process
var explicitBenFranklinDecisionPrompt = """
Apply the following steps IN ORDER of the Ben Franklin Decision Framework to the Question below:
1) Frame a decision that has two options (Yes or a No)
2) Divide an area into two competing halves: a "Pro" side and "Con" side
3) Label the top of one side "Pro" (for) and the other "Con" (against)
4) Under each respective side, list a maximum of 5 reasons or arguments for each option. If there less than 5 reasons, only list the actual number of reasons there are for each side. Do not list filler reasons to reach 5 if there are not actually 5 reasons for that side.
5) Consider the weight of each reason or argument. This is done by taking reasons/arguments that are of similar value (weight) and crossing them off of the other competing half. Multiple reasons can be combined from one side to form a "subjective" value (weight) to balance out the other half. (For example, two medium "Pro" reasons might add up to an equal value of a single important "Con" reason)
6) Determine the relative importance of each reason or argument. This is done by taking reasons/arguments that are of similar value (weight) and crossing them off of the other competing half. Multiple reasons can be combined from one side to form a "subjective" value (weight) to balance out the other half. (For example, two medium "Pro" reasons might add up to an equal value of a single important "Con" reason)
7) The side with the most remaining reasons is the option one should select for the decision in question

IMPORTANT: ALWAYS recommend a decision based on the side with the most remaining reasons, even if the reasons are of lesser value than the other side!

Question: Should I take a luxury family vacation?
""";

List<ChatMessage> chatMessagesLuxuryVacationDecisionImprovedWithBackground =
[
    new(ChatRole.System, systemDecisionPrompt),
    new(ChatRole.User, backroundInformation),
    new(ChatRole.User, explicitBenFranklinDecisionPrompt),
];

// Define reasoning options for the chat completion.
// Note: For speed optimization, reasoning effort is set to None.
var reasoningOptions = new ReasoningOptions
{
    Effort = ReasoningEffort.None
};

// Note: Different OSS models have different capabilities and settings
// Gemma-4-26B Model Card Recommendations: https://huggingface.co/google/gemma-4-26B-A4B#1-sampling-parameters 
ChatOptions chatOptions = new()
{
    Temperature = 1.0f,
    TopP = 0.95f,
    TopK = 64, 
    Reasoning = reasoningOptions
};

// Execute the chat messages through the Microsoft.Extensions.AI IChatClient API
ChatResponse chatLuxuryVacationDecisionImprovedWithBackgroundResponse = 
    await localAIChatClient.GetResponseAsync(chatMessagesLuxuryVacationDecisionImprovedWithBackground, chatOptions);
var chatLuxuryVacationDecisionImprovedWithBackgroundResponseText = chatLuxuryVacationDecisionImprovedWithBackgroundResponse.Text;

// Render the response string as Markdown
chatLuxuryVacationDecisionImprovedWithBackgroundResponseText.DisplayAs("text/markdown");

| Step | Analysis Component | Details |
| :--- | :--- | :--- |
| 1 | Decision Frame | Should I take a luxury family vacation? (Yes or No) |
| 2-4 | Pro Argument List | 1. Immediate relief from work-related stress and burnout. <br> 2. Opportunity to utilize recent promotion/bonus for meaningful family bonding. <br> 3. Mental reset after a year of no vacation. |
| 2-4 | Con Argument List | 1. High cost reduces capital available for a new car lease/purchase. <br> 2. Potential financial strain if work stress leads to further career changes. |
| 5-6 | Weighting and Cancellation Process | Two medium Pro reasons (Stress relief + Mental reset) are balanced against one significant Con reason (Car replacement cost). One remaining Pro reason (Family bonding/Bonus use) is balanced against one significant Con reason (Financial strain). |
| 7 | Final Decision | **Yes** (The Pro side has more remaining reasons/arguments after the cancellation process). |

Notice how providing personal family background information changes the entire dynamic of the information used in the decision framework and how it influences the recommended decision. The decision process is more specific not only to the scenario, but also it provides contextual background information. This makes the decision process more personalized and potentially much more accurate! 

---
### Step 6 - Higher Reasoning Effort for High-Stakes Decisions  

Let's assume that this a very high-stakes decision. In this case we would prefer to have the AI system spend more time on the decision. Let's turn reasoning on to see how this changes the answer. 

In [8]:
// Define reasoning options for the chat completion.
// Note: For additional reasoning capabilities, reasoning effort is set to High
var reasoningOptions = new ReasoningOptions
{
    Effort = ReasoningEffort.High
};

// Note: Different OSS models have different capabilities and settings
// Gemma-4-26B Model Card Recommendations: https://huggingface.co/google/gemma-4-26B-A4B#1-sampling-parameters 
ChatOptions chatOptions = new()
{
    Temperature = 1.0f,
    TopP = 0.95f,
    TopK = 64, 
    Reasoning = reasoningOptions
};

// Execute the chat messages through the Microsoft.Extensions.AI IChatClient API
ChatResponse chatLuxuryVacationDecisionImprovedWithBackgroundResponse = 
    await localAIChatClient.GetResponseAsync(chatMessagesLuxuryVacationDecisionImprovedWithBackground, chatOptions);
var chatLuxuryVacationDecisionImprovedWithBackgroundResponseText = chatLuxuryVacationDecisionImprovedWithBackgroundResponse.Text;

// Render the response string as Markdown
chatLuxuryVacationDecisionImprovedWithBackgroundResponseText.DisplayAs("text/markdown");

| Step | Decision Analysis: Should I take a luxury family vacation? |
| :--- | :--- |
| 1. Frame the Decision | **Decision:** Yes (Take the vacation) or No (Do not take the vacation). |
| 2 & 3. Divide and Label Sides | **Pro (For)** vs. **Con (Against)** |
| 4. List of Reasons | **Pro:** <br>1. Mitigate burnout and work-related stress (High weight) <br>2. Reconnect with family after a year of long hours (Medium weight) <br>3. Celebrate career advancement and bonus (Low weight) <br><br>**Con:** <br>1. Diverts capital from secondary home investment (High weight) <br>2. Reduces budget for upcoming car replacement (Medium weight) |
| 5. Weighting Process | **Balancing High Weights:** Pro Reason 1 is matched and canceled by Con Reason 1. <br>**Balancing Medium Weights:** Pro Reason 2 is matched and canceled by Con Reason 2. |
| 6. Relative Importance | **Remaining Pro reasons:** 1 (Reason #3) <br>**Remaining Con reasons:** 0 |
| 7. Final Decision | **YES** <br>(The decision is to take the luxury family vacation because it is the side with the most remaining reasons.) |

---
### Step 7 - Open Source AI & Commercial AI Providers  

In .NET you can include mutliple AI chatclients with different service providers. This allows for hybrid workflows from a single service provider:
* Capability Optimizations: Use SLMs for domain specific tasks and LLMs for complex decision reasoning
* Decision Optimizations: Apply an (ensemble) decision self-consitency pattern  
* Capacity Optimizations: Splitting functions, plugins, personas or agents across different AI services

In [9]:
// Import the required NuGet configuration packages
#r "nuget: Azure.AI.OpenAI, 2.9.0-beta.1"
#r "nuget: Microsoft.Extensions.AI.OpenAI, 10.5.0"
#r "nuget: Microsoft.Extensions.Configuration.Json, 10.0.6"
#r "nuget: Microsoft.Extensions.DependencyInjection, 10.0.6"
#r "nuget: OpenAI, 2.10.0"

using Azure.AI.OpenAI;
using Microsoft.Extensions.AI;
using Microsoft.Extensions.Configuration.Json;
using Microsoft.Extensions.Configuration;
using Microsoft.Extensions.DependencyInjection;
using OpenAI;
using System.ClientModel;
using System.ComponentModel;
using System.IO;

// Load the configuration settings from the local.settings.json and secrets.settings.json files
// The secrets.settings.json file is used to store sensitive information such as API keys
var configurationBuilder = new ConfigurationBuilder()
    .SetBasePath(Directory.GetCurrentDirectory())
    .AddJsonFile("local.settings.json", optional: true, reloadOnChange: true)
    .AddJsonFile("secrets.settings.json", optional: true, reloadOnChange: true);
var config = configurationBuilder.Build();

// Retrieve the configuration settings for the Azure OpenAI service
var azureOpenAIEndpoint = config["AzureOpenAI:Endpoint"];
var azureOpenAIAPIKey = config["AzureOpenAI:APIKey"];
var azureOpenAIModelDeploymentName = config["AzureOpenAI:ModelDeploymentName"];

// Retrieve the configuration settings for the local OpenAI-compatible service
var localApiKey = "not_needed_for_lmstudio_ollama_etc"; // local LLMs do not require an API key
var localUrl = "http://10.0.0.61:1234/v1/"; // this can be localhost or an IP address
var localModelName = "google/gemma-4-26b-a4b"; // Another Option: "openai/gpt-oss-20b";

// Create a service collection for dependency injection
var services = new ServiceCollection();

// Add Cloud AI Client Configuration
var apiKeyCredential = new ApiKeyCredential(azureOpenAIAPIKey);
var azureOpenAIClient = new AzureOpenAIClient(new Uri(azureOpenAIEndpoint), apiKeyCredential);
var cloudChatClient = azureOpenAIClient.GetChatClient(azureOpenAIModelDeploymentName).AsIChatClient();

// Add Local OpenAI Client Configuration
var apiCredentials = new ApiKeyCredential(localApiKey);
var openAIClientOptions = new OpenAIClientOptions
{
    Endpoint = new Uri(localUrl)
};
// Create a local AI client 
var localAIClient = new OpenAIClient(apiCredentials, openAIClientOptions);

// Wrap the OpenAI-compatible chat client with the Microsoft.Extensions.AI abstraction
IChatClient localAIChatClient = localAIClient.GetChatClient(localModelName)
    .AsIChatClient();

// Add cloud and local AI clients to the service collection for dependency injection
services.AddKeyedSingleton<IChatClient>("cloudAI", cloudChatClient);
services.AddKeyedSingleton<IChatClient>("localAI", localAIChatClient);

var provider = services.BuildServiceProvider();

// Reference the cloud and local AI clients from the service provider
var cloud = provider.GetRequiredKeyedService<IChatClient>("cloudAI");
var local = provider.GetRequiredKeyedService<IChatClient>("localAI");

Installed Packages Azure.AI.OpenAI, 2.9.0-beta.1 Microsoft.Extensions.AI.OpenAI, 10.5.0 Microsoft.Extensions.Configuration.Json, 10.0.6 Microsoft.Extensions.DependencyInjection, 10.0.6 OpenAI, 2.10.0


warning CS1701: Assuming assembly reference 'OpenAI, Version=2.9.1.0, Culture=neutral, PublicKeyToken=b4187f3e65366280' used by 'Azure.AI.OpenAI' matches identity 'OpenAI, Version=2.10.0.0, Culture=neutral, PublicKeyToken=b4187f3e65366280' of 'OpenAI', you may need to supply runtime policy

warning CS1701: Assuming assembly reference 'System.ClientModel, Version=1.9.0.0, Culture=neutral, PublicKeyToken=92742159e12e44c8' used by 'Azure.AI.OpenAI' matches identity 'System.ClientModel, Version=1.10.0.0, Culture=neutral, PublicKeyToken=92742159e12e44c8' of 'System.ClientModel', you may need to supply runtime policy

warning CS1701: Assuming assembly reference 'System.ClientModel, Version=1.9.0.0, Culture=neutral, PublicKeyToken=92742159e12e44c8' used by 'Azure.Core' matches identity 'System.ClientModel, Version=1.10.0.0, Culture=neutral, PublicKeyToken=92742159e12e44c8' of 'System.ClientModel', you may need to supply runtime policy

warning CS1701: Assuming assembly reference 'OpenAI, Ver